# 국민여가활동조사 선호모형 EDA 및 전처리

- 국민여가활동조사 2021~2025 자료를 선호 예측 학습용으로 정리함.
- `가장 만족스러운 여가활동` 1~3순위를 목표 라벨로 사용함.
- 원자료 구조, 입력 변수, 목표 라벨, 중분류 매핑 결과를 점검함.

## 분석 환경 및 경로 설정

- `notebooks/preference` 기준 분석 경로를 설정함.
- 원자료, 매핑자료, 전처리 산출물 경로를 생성함.

In [ ]:
import pandas as pd
import numpy as np
import re
from pathlib import Path

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 120)

BASE_PATH = Path.cwd().resolve()

PREFERENCE_PATH = None
for path in [BASE_PATH, *BASE_PATH.parents]:
    if (path / "notebooks" / "preference" / "data").exists():
        PREFERENCE_PATH = path / "notebooks" / "preference"
        break

if PREFERENCE_PATH is None:
    raise FileNotFoundError("preference ?? ??? ?? ?????.")
DATA_PATH = PREFERENCE_PATH / "data"
SOURCE_PATH = DATA_PATH / "source"
MAPPING_PATH = DATA_PATH / "mapping"
PROCESSED_PATH = DATA_PATH / "processed" / "satisfaction"
PROCESSED_PATH.mkdir(parents=True, exist_ok=True)

SURVEY_PATH = SOURCE_PATH / "leisure_activity_survey_2021_2025_selected_columns.csv"
MAPPING_WORKBOOK_PATH = MAPPING_PATH / "leisure_activity_to_mnc_category_mapping.xlsx"

print("preference 경로:", PREFERENCE_PATH)
print("원자료 경로:", SURVEY_PATH)
print("매핑 참고자료 경로:", MAPPING_WORKBOOK_PATH)
print("산출물 경로:", PROCESSED_PATH)

## 원자료 불러오기 및 기본 구조 확인

- 2021~2025 국민여가활동조사 분석용 선택 칼럼 파일을 불러옴.
- `응답자_ID`를 순번 기반으로 생성함.
- 자료 구조, 조사년도 분포, 응답자 ID 중복 여부를 확인함.

In [ ]:
survey = pd.read_csv(SURVEY_PATH, encoding="utf-8-sig")

survey = survey.copy()
survey["응답자_ID"] = [
    f"RESP_{i:05d}" for i in range(1, len(survey) + 1)
]

print("원자료 구조:", survey.shape)
print("응답자_ID 중복:", survey["응답자_ID"].duplicated().sum())
print("\n조사년도 분포")
print(survey["조사년도"].value_counts().sort_index())

display(survey.head())

## 입력 특성 변수 점검

- 격자 적용 가능 변수와 후보 확장 변수를 구분함.
- 입력 후보 변수의 존재 여부와 결측치를 확인함.

In [ ]:
direct_feature_cols = [
    "성별",
    "연령",
    "17개 시도",
    "지역규모",
    "조사년도",
    "최종가중치",
]

candidate_feature_cols = [
    "학력",
    "가구소득",
    "장애여부",
    "여가활동을 위한 월평균 지출액",
    "적절하다고 생각하는 월평균 여가비용",
    "평일 하루 평균 여가시간",
    "휴일 하루 평균 여가시간",
    "생활권 내 공공문화여가시설 이용 충분도",
    "생활권 내 공공문화여가시설 이용 여부",
    "생활권 내 공공문화여가시설 만족도",
    "여가활동 제약요인_시간부족",
    "여가활동 제약요인_경제적 지출 부담",
    "여가활동 제약요인_여가활동 경험 부족",
    "여가활동 제약요인_여가 정보 부족",
    "여가활동 제약요인_질병 및 장애",
    "여가활동 제약요인_여가 동반자 없음",
    "여가활동 제약요인_여가시설 접근성 부족",
    "여가활동 제약요인_여가프로그램 부족",
]

feature_cols = direct_feature_cols + candidate_feature_cols
missing_feature_cols = [col for col in feature_cols if col not in survey.columns]

print("입력 후보 변수 수:", len(feature_cols))
print("누락된 입력 후보 변수:", missing_feature_cols)

feature_check = (
    survey[feature_cols]
    .isna()
    .sum()
    .reset_index(name="결측치")
    .rename(columns={"index": "칼럼명"})
)
feature_check["결측률"] = feature_check["결측치"] / len(survey)

display(feature_check)

## 목표변수 결측 및 연도별 사용 가능성 점검

- `가장 만족스러운 여가활동` 1~3순위 결측률을 확인함.
- `향후 희망하는 여가활동` 1~3순위의 조사년도별 결측률을 확인함.

In [ ]:
satisfaction_cols = [
    "가장 만족스러운 여가활동 1순위",
    "가장 만족스러운 여가활동 2순위",
    "가장 만족스러운 여가활동 3순위",
]

hope_cols = [
    "향후 희망하는 여가활동 1순위",
    "향후 희망하는 여가활동 2순위",
    "향후 희망하는 여가활동 3순위",
]

target_check = (
    survey[satisfaction_cols + hope_cols]
    .isna()
    .sum()
    .reset_index(name="결측치")
    .rename(columns={"index": "칼럼명"})
)
target_check["결측률"] = target_check["결측치"] / len(survey)

print("목표변수 결측 점검")
display(target_check)

year_target_check = (
    survey
    .groupby("조사년도")[satisfaction_cols + hope_cols]
    .apply(lambda x: x.isna().mean())
)

print("\n조사년도별 목표변수 결측률")
display(year_target_check)

## 문화누리 중분류 매핑표 정리

- 여가활동 코드 1~88을 문화누리 가맹점 중분류로 매핑함.
- `스포츠 관람` 표기를 `스포츠관람`으로 통일함.
- `교통수단`, `숙박`, `여행사`, `분류범위외`를 제외 분류로 지정함.

In [ ]:
category_code_dict = {
    "도서": [65, 66, 78],
    "음악": [76, 77],
    "영상": [7, 17, 74, 75],
    "공연": [3, 4, 5, 6, 8],
    "미술": [1, 2, 14],
    "문화체험": [9, 10, 11, 12, 13, 15, 50, 51, 69],
    "교통수단": [48],
    "여행사": [42],
    "관광지": [38, 39, 40, 41, 43, 44, 45, 46, 47, 55, 72],
    "스포츠관람": [16, 18, 19],
    "체육용품": [35],
    "체육시설": [20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 36, 37, 56],
    "분류범위외": [49, 52, 53, 54, 57, 58, 59, 60, 61, 62, 63, 64, 67, 68, 70, 71, 73, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88],
}

exclude_categories = ["교통수단", "숙박", "여행사", "분류범위외"]

valid_categories = [
    "도서",
    "문화체험",
    "음악",
    "영상",
    "체육시설",
    "체육용품",
    "미술",
    "공연",
    "스포츠관람",
    "관광지",
]

activity_cols = [
    col for col in survey.columns
    if col.startswith("한 번 이상 참여한 여가활동 -")
]

activity_rows = []

for col in activity_cols:
    code_match = re.search(r"\((\d+)\)", col)
    
    if code_match is None:
        continue
    
    activity_code = int(code_match.group(1))
    activity_name = re.sub(r"^한 번 이상 참여한 여가활동 - \(\d+\)\s*", "", col)
    
    activity_rows.append({
        "활동코드": activity_code,
        "여가활동명": activity_name
    })

activity_mapping = pd.DataFrame(activity_rows).sort_values("활동코드")

code_to_category = {}

for category, code_list in category_code_dict.items():
    for activity_code in code_list:
        code_to_category[activity_code] = category

activity_mapping["중분류"] = activity_mapping["활동코드"].map(code_to_category)
activity_mapping["학습타깃사용여부"] = activity_mapping["중분류"].isin(valid_categories)

print("여가활동 코드 수:", activity_mapping["활동코드"].nunique())
print("매핑 누락 코드 수:", activity_mapping["중분류"].isna().sum())
print("중복 매핑 코드 수:", activity_mapping["활동코드"].duplicated().sum())
print("\n중분류별 활동 코드 수")
print(activity_mapping["중분류"].value_counts(dropna=False))

display(activity_mapping.head(20))

## 만족 여가활동 value와 매핑표 대조

- 만족 1~3순위에 등장한 활동 코드를 추출함.
- 관측 활동 코드와 중분류 매핑표의 누락 여부를 확인함.

In [ ]:
observed_target_values = (
    pd.concat([survey[col] for col in satisfaction_cols], ignore_index=True)
    .dropna()
    .astype(int)
    .sort_values()
    .unique()
)

mapping_code_set = set(activity_mapping["활동코드"])
unmapped_observed_values = sorted(set(observed_target_values) - mapping_code_set)

print("만족 1~3순위 관측 코드 수:", len(observed_target_values))
print("만족 1~3순위 매핑 누락 코드:", unmapped_observed_values)

target_value_count = (
    pd.concat([survey[col] for col in satisfaction_cols], ignore_index=True)
    .dropna()
    .astype(int)
    .value_counts()
    .sort_index()
    .reset_index()
)
target_value_count.columns = ["활동코드", "관측수"]
target_value_count = target_value_count.merge(activity_mapping, on="활동코드", how="left")

display(target_value_count.head(20))

## 만족 순위 데이터 전처리

- 만족 1~3순위 활동 코드를 문화누리 중분류로 변환함.
- 제외 분류를 제거함.
- 남은 중분류를 앞 순위부터 재정렬함.
- 동일 중분류 반복 응답은 첫 등장만 유지함.

In [ ]:
def get_category(value):
    if pd.isna(value):
        return np.nan
    
    try:
        activity_code = int(value)
    except Exception:
        return np.nan
    
    return code_to_category.get(activity_code, np.nan)


def preprocess_satisfaction_row(row):
    raw_categories = [
        get_category(row[col])
        for col in satisfaction_cols
    ]
    
    valid_rank_categories = []
    excluded_categories = []
    duplicated_categories = []
    
    for category in raw_categories:
        if pd.isna(category):
            continue
        
        if category in exclude_categories:
            excluded_categories.append(category)
            continue
        
        if category not in valid_categories:
            excluded_categories.append(category)
            continue
        
        if category in valid_rank_categories:
            duplicated_categories.append(category)
            continue
        
        valid_rank_categories.append(category)
    
    result = {}
    
    for rank in range(3):
        result[f"만족_원중분류_{rank + 1}순위"] = raw_categories[rank]
        result[f"만족_유효중분류_{rank + 1}순위"] = (
            valid_rank_categories[rank]
            if rank < len(valid_rank_categories)
            else np.nan
        )
    
    result["만족_유효순위수"] = len(valid_rank_categories)
    result["제외분류_제거수"] = len(excluded_categories)
    result["중복중분류_제거수"] = len(duplicated_categories)
    result["제외분류목록"] = ", ".join(excluded_categories)
    result["중복중분류목록"] = ", ".join(duplicated_categories)
    
    return pd.Series(result)


preprocessed = survey.copy()
target_processed = preprocessed.apply(preprocess_satisfaction_row, axis=1)
preprocessed = pd.concat([preprocessed, target_processed], axis=1)

preprocessed["학습사용여부"] = preprocessed["만족_유효순위수"] > 0

print("전처리 전 응답자 수:", len(preprocessed))
print("학습 사용 가능 응답자 수:", preprocessed["학습사용여부"].sum())
print("학습 제외 응답자 수:", (~preprocessed["학습사용여부"]).sum())
print("\n제외분류 제거수 분포")
print(preprocessed["제외분류_제거수"].value_counts().sort_index())
print("\n중복중분류 제거수 분포")
print(preprocessed["중복중분류_제거수"].value_counts().sort_index())

## 전처리 후 데이터 품질 점검

- 학습 사용 가능 응답자 수를 확인함.
- 1순위 유효 중분류 결측 여부를 확인함.
- 1순위 유효 중분류와 조사년도별 분포를 확인함.

In [ ]:
training_wide = preprocessed[preprocessed["학습사용여부"]].copy()

rank1_col = "만족_유효중분류_1순위"

print("학습 가능 응답자 테이블 구조:", training_wide.shape)
print("1순위 유효 중분류 결측:", training_wide[rank1_col].isna().sum())
print("응답자_ID 중복:", training_wide["응답자_ID"].duplicated().sum())

print("\n1순위 유효 중분류 분포")
print(training_wide[rank1_col].value_counts())

print("\n조사년도별 1순위 유효 중분류 분포")
display(pd.crosstab(training_wide["조사년도"], training_wide[rank1_col]))

## 만족 순위 학습용 base 테이블 생성

- 응답자 단위 학습 base 테이블을 생성함.
- 입력 변수와 만족 유효 1~3순위 칼럼을 유지함.
- 유효 순위 수, 제외 분류 제거 수, 중복 중분류 제거 수를 함께 저장함.

In [ ]:
rank_base_cols = (
    ["응답자_ID"]
    + feature_cols
    + [
        "만족_유효중분류_1순위",
        "만족_유효중분류_2순위",
        "만족_유효중분류_3순위",
        "만족_유효순위수",
        "제외분류_제거수",
        "중복중분류_제거수",
    ]
)

rank_base = training_wide[rank_base_cols].copy()

rank_cols = [
    "만족_유효중분류_1순위",
    "만족_유효중분류_2순위",
    "만족_유효중분류_3순위",
]

rank_category_count = (
    pd.concat([rank_base[col] for col in rank_cols], ignore_index=True)
    .dropna()
    .value_counts()
)

print("Rank base 테이블 구조:", rank_base.shape)
print("응답자_ID 중복:", rank_base["응답자_ID"].duplicated().sum())
print("\n유효 순위 수 분포")
print(rank_base["만족_유효순위수"].value_counts().sort_index())
print("\n1~3순위 전체 중분류 분포")
print(rank_category_count)

display(rank_base.head())

## 전처리 산출물 저장

- 만족 순위 학습용 base 테이블을 저장함.
- 여가활동 코드-문화누리 중분류 매핑표를 저장함.

In [ ]:
rank_base_output_path = PROCESSED_PATH / "ml_preference_satisfaction_rank_base.csv"
mapping_output_path = PROCESSED_PATH / "ml_activity_category_mapping.csv"

old_output_paths = [
    PROCESSED_PATH / "ml_preference_satisfaction_preprocessed_all.csv",
    PROCESSED_PATH / "ml_preference_satisfaction_training_wide.csv",
    PROCESSED_PATH / "ml_preference_satisfaction_training_long.csv",
    rank_base_output_path,
    mapping_output_path,
]

for output_path in old_output_paths:
    output_path.unlink(missing_ok=True)


def save_csv(data, output_path):
    with output_path.open("w", encoding="utf-8-sig", newline="") as file:
        data.to_csv(file, index=False)


save_csv(rank_base, rank_base_output_path)
save_csv(activity_mapping, mapping_output_path)

print("저장 완료")
print(rank_base_output_path)
print(mapping_output_path)